# Preprocessing

## 1.   Cleaning

In [10]:
import os
import sys

sys.path.insert(0, os.path.abspath("../../src"))

In [15]:
from preprocessing.cleaning import (
    clean_text,
    demojize,
    normalize_repeated_punctuation,
    normalize_unicode,
    process_hashtags,
    reduce_repeated_chars,
    remove_urls,
)

ModuleNotFoundError: No module named 'preprocessing'

In [18]:
nfc = "café"
nfd = "caféine"
normalize_unicode(nfd)

'caféine'

In [ ]:
text = (
    "Livraison lente cette fois-ci voir "
    "https://claude.ai/chat/49137f24-82cf-40a2-90a7-91840e86c17d svp "
)
remove_urls(text).strip()

'Livraison lente cette fois-ci voir  svp'

In [36]:
text = "J'adore #Python et #MachineLearning"
process_hashtags(text)

"J'adore Python et Machine Learning"

In [46]:
result = demojize("Livraison rapide 👍 or 🎉  or 😡")
result

'Livraison rapide  __EMOJI_thumbs_up__  or  __EMOJI_party_popper__   or  __EMOJI_enraged_face__ '

In [50]:
raw = (
    "@GlobaTrend Livraison sooooo lente !!!! 😡 mais le produit "
    "est top #GreatService, voir https://amazon.fr/produit/123 pour + d'infos"
)
clean_text(raw)

"livraison soo lente ! __emoji_enraged_face__ mais le produit est top great service, voir pour + d'infos"

In [51]:
# test reduce_repeated_chars
text = "soooooo goooood"
reduce_repeated_chars(text)

'soo good'

In [52]:
# test repeated punctuation
text = "What a great product!!!!!!!"
normalize_repeated_punctuation(text)

'What a great product!'

## 2.   tokenization

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../../src"))

from preprocessing.cleaning import (
    clean_text,
    reduce_repeated_chars,
)

In [ ]:
def whitespace_tokenize(text: str) -> list[str]:
    return text.split()


# test whitespace_tokenize
text = "This is a test sentence."
whitespace_tokenize(text)

['This', 'is', 'a', 'test', 'sentence.']

In [ ]:
from nltk.tokenize import word_tokenize

text = "This is a test sentence!!!!!!!!"
word_tokenize(reduce_repeated_chars(text), language="english")

['This', 'is', 'a', 'test', 'sentence', '!', '!']

In [ ]:
from nltk.tokenize import sent_tokenize

texte = (
    "Dr. Smith recommended this product. It arrived in 9.5 days, "
    "e.g. slower than expected. I'm not  happy!!!! 😡 "
)

# Découpage naïf sur le point
naif = clean_text(texte).split(".")

# Découpage correct avec Punkt (NLTK)
correct = sent_tokenize(clean_text(texte), language="english")

print("Naïf (split sur '.'):")
for i, s in enumerate(naif):
    print(f"  {i}: {s!r}")

print()
print("Correct (sent_tokenize):")
for i, s in enumerate(correct):
    print(f"  {i}: {s!r}")

Naïf (split sur '.'):
  0: 'dr'
  1: ' smith recommended this product'
  2: ' it arrived in 9'
  3: '5 days, e'
  4: 'g'
  5: ' slower than expected'
  6: " i'm not happy! __emoji_enraged_face__"

Correct (sent_tokenize):
  0: 'dr. smith recommended this product.'
  1: 'it arrived in 9.5 days, e.g.'
  2: 'slower than expected.'
  3: "i'm not happy!"
  4: '__emoji_enraged_face__'


In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize

texte = (
    "Dr. Smith recommended this product. It arrived in 2.5 days, "
    "e.g. faster than expected. I'm happy! However, the packaging was "
    "damaged. See https://example.com for details."
)


def tokenize_document(text, language="english"):
    phrases = sent_tokenize(text, language=language)
    return [word_tokenize(phrase, language=language) for phrase in phrases]


resultat = tokenize_document(texte)

print(f"{len(resultat)} phrases trouvées:\n")
for i, phrase_tokens in enumerate(resultat):
    print(f"Phrase {i}: {phrase_tokens}")

6 phrases trouvées:

Phrase 0: ['Dr.', 'Smith', 'recommended', 'this', 'product', '.']
Phrase 1: ['It', 'arrived', 'in', '2.5', 'days', ',', 'e.g', '.']
Phrase 2: ['faster', 'than', 'expected', '.']
Phrase 3: ['I', "'m", 'happy', '!']
Phrase 4: ['However', ',', 'the', 'packaging', 'was', 'damaged', '.']
Phrase 5: ['See', 'https', ':', '//example.com', 'for', 'details', '.']


## 3.   Normalization


### Steaming

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()


def stem_tokens(tokens: list[str]) -> list[str]:
    return [stemmer.stem(tok) for tok in tokens]

In [ ]:
# test stem_tokens
tokens = ["running", "jumps", "easily", "fairly", "happiness"]
stemmed = stem_tokens(tokens)
print("Original tokens:", tokens)
print("Stemmed tokens:", stemmed)

Original tokens: ['running', 'jumps', 'easily', 'fairly', 'happiness']
Stemmed tokens: ['run', 'jump', 'easili', 'fairli', 'happi']


### Lemmatization

In [ ]:
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()


def _penn_to_wordnet_pos(penn_tag: str) -> str:
    if penn_tag.startswith("J"):
        return "a"
    if penn_tag.startswith("V"):
        return "v"
    if penn_tag.startswith("R"):
        return "r"
    return "n"


def lemmatize_tokens(tokens: list[str], use_pos: bool = True) -> list[str]:
    if not use_pos:
        return [lemmatizer.lemmatize(tok) for tok in tokens]
    tagged = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(tok, pos=_penn_to_wordnet_pos(tag)) for tok, tag in tagged
    ]

In [ ]:
# test lemmatize_tokens
tokens = ["running", "jumps", "easily", "fairly", "wanted"]
lemmatized = lemmatize_tokens(tokens)
print("Original tokens:", tokens)
print("Lemmatized tokens:", lemmatized)

Original tokens: ['running', 'jumps', 'easily', 'fairly', 'wanted']
Lemmatized tokens: ['run', 'jump', 'easily', 'fairly', 'want']


In [ ]:
from nltk.corpus import stopwords

_STOPWORDS_CACHE = {}


def get_stopwords(language: str = "english") -> set[str]:
    if language not in _STOPWORDS_CACHE:
        _STOPWORDS_CACHE[language] = set(stopwords.words(language))
    return _STOPWORDS_CACHE[language]


def remove_stopwords(tokens: list[str], language: str = "english") -> list[str]:
    sw = get_stopwords(language)
    return [tok for tok in tokens if tok.lower() not in sw]

In [ ]:
# test stopwords removal
tokens = ["This", "is", "a", "test", "sentence", "."]
filtered_tokens = remove_stopwords(tokens)
print("Original tokens:", tokens)
print("Filtered tokens:", filtered_tokens)

Original tokens: ['This', 'is', 'a', 'test', 'sentence', '.']
Filtered tokens: ['test', 'sentence', '.']


## 4.   Linguistic


In [ ]:
def pos_tag_tokens(tokens: list[str]) -> list[tuple[str, str]]:
    return pos_tag(tokens)


def extract_noun_phrases(tokens: list[str]) -> list[str]:
    tagged = pos_tag_tokens(tokens)
    phrases, current = [], []
    for word, tag in tagged:
        if tag.startswith("NN"):
            current.append(word)
        else:
            if current:
                phrases.append(" ".join(current))
                current = []
    if current:
        phrases.append(" ".join(current))
    return phrases

In [ ]:
# test pos_tag_tokens
tokens = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]
print(pos_tag_tokens(tokens))

[('The', 'DT'), ('quick', 'JJ'), ('brown', 'NN'), ('fox', 'NN'), ('jumps', 'VBZ'), ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dog', 'NN')]


In [ ]:
# test exxtra_noun_phrases
extract_noun_phrases(tokens)

['brown fox', 'dog']

In [ ]:
from nltk import ne_chunk
from nltk.tree import Tree


def named_entities_nltk(tokens: list[str]) -> list[tuple[str, str]]:
    tagged = pos_tag(tokens)
    tree = ne_chunk(tagged)
    entities = []
    for subtree in tree:
        if isinstance(subtree, Tree):
            entity_text = " ".join(tok for tok, _ in subtree.leaves())
            entities.append((entity_text, subtree.label()))
    return entities

In [ ]:
from nltk import word_tokenize

exemples = [
    "I bought this Samsung phone from Amazon last week",
    "John from customer service was very rude to me",
    "I ordered it while living in Paris",
]

for phrase in exemples:
    tokens = word_tokenize(phrase)
    entities = named_entities_nltk(tokens)
    print(f"{phrase}\n  -> entités : {entities}\n")

I bought this Samsung phone from Amazon last week
  -> entités : [('Samsung', 'ORGANIZATION'), ('Amazon', 'GPE')]

John from customer service was very rude to me
  -> entités : [('John', 'PERSON')]

I ordered it while living in Paris
  -> entités : [('Paris', 'GPE')]



In [ ]:
import spacy

_SPACY_MODELS = {}


def _load_spacy_model(lang: str = "en"):
    model_names = {
        "en": "en_core_web_sm",
        "es": "es_core_news_sm",
        "de": "de_core_news_sm",
        "fr": "fr_core_news_sm",
    }
    if lang not in _SPACY_MODELS:
        _SPACY_MODELS[lang] = spacy.load(model_names[lang])
    return _SPACY_MODELS[lang]


def dependency_parse(text: str, lang: str = "en") -> list[dict]:
    nlp = _load_spacy_model(lang)
    doc = nlp(text)
    return [
        {"token": tok.text, "dep": tok.dep_, "head": tok.head.text, "pos": tok.pos_}
        for tok in doc
    ]

In [ ]:
phrase = "the product is great but delivery was slow"
print(f"Phrase analysée : {phrase!r}\n")

resultats = dependency_parse(phrase, lang="en")

for r in resultats:
    print(f"  {r['token']:12} dep={r['dep']:10} head={r['head']:10} pos={r['pos']}")

print("\nCe qu'il faut vérifier :")
print('  - "slow" doit avoir dep=acomp et head=was')
print('  - "delivery" doit avoir dep=nsubj et head=was')
print('  -> ça confirme que "slow" se rattache bien à "delivery" via "was",')

Phrase analysée : 'the product is great but delivery was slow'



  the          dep=det        head=product    pos=DET
  product      dep=nsubj      head=is         pos=NOUN
  is           dep=ROOT       head=is         pos=AUX
  great        dep=acomp      head=is         pos=ADJ
  but          dep=cc         head=is         pos=CCONJ
  delivery     dep=nsubj      head=was        pos=NOUN
  was          dep=conj       head=is         pos=AUX
  slow         dep=acomp      head=was        pos=ADJ

Ce qu'il faut vérifier :
  - "slow" doit avoir dep=acomp et head=was
  - "delivery" doit avoir dep=nsubj et head=was
  -> ça confirme que "slow" se rattache bien à "delivery" via "was",


## 5.   Handling Negation



In [2]:
NEGATION_WORDS = {"not", "no", "never", "n't", "cannot", "nothing", "neither", "nor"}
PUNCTUATION = {".", "!", "?", ",", ";", ":"}


def mark_negation_scope(tokens):
    result = []
    negating = False
    for token in tokens:
        if token.lower() in NEGATION_WORDS:
            result.append(token)
            negating = True
        elif token in PUNCTUATION:
            result.append(token)
            negating = False  # la négation s'arrête à la ponctuation
        elif negating:
            result.append(token + "_NEG")
        else:
            result.append(token)
    return result


exemples = [
    ["delivery", "was", "not", "good"],
    [
        "I",
        "do",
        "n't",
        "like",
        "this",
        "product",
        ",",
        "but",
        "the",
        "service",
        "was",
        "great",
    ],
]

for tokens in exemples:
    print("Original :", tokens)
    print("Marqué   :", mark_negation_scope(tokens))
    print()

Original : ['delivery', 'was', 'not', 'good']
Marqué   : ['delivery', 'was', 'not', 'good_NEG']

Original : ['I', 'do', "n't", 'like', 'this', 'product', ',', 'but', 'the', 'service', 'was', 'great']
Marqué   : ['I', 'do', "n't", 'like_NEG', 'this_NEG', 'product_NEG', ',', 'but', 'the', 'service', 'was', 'great']



In [6]:
from preprocessing.normalization import remove_stopwords

tokens = ["delivery", "was", "not", "good"]
marked = mark_negation_scope(tokens)
remove_stopwords(marked)

ModuleNotFoundError: No module named 'preprocessing'